<a href="https://colab.research.google.com/github/PujaSridhar/finsight/blob/main/notebooks/01_build_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FinSight · 1. Build the earnings dataset

Downloads quarterly earnings press releases (Form 8-K, Item 2.02, Exhibit 99.1) from SEC EDGAR and labels each
one with the figures the company later filed in XBRL (revenue, GAAP net income, GAAP diluted EPS, period end).
A release is kept only if those figures appear in the text the model will read. Splits are by company.

CPU only. A 5-company check takes a couple of minutes; the full 150-company build takes roughly 30-60 minutes,
because the SEC allows at most 10 requests a second. Everything is cached on Google Drive, so a re-run is quick.

In [ ]:
# Clone the repo and install dependencies (~2 min)
REPO_BRANCH = "main"   # to try a branch before merging it, put its name here
!git clone -q -b $REPO_BRANCH https://github.com/PujaSridhar/finsight /content/finsight 2>/dev/null || (cd /content/finsight && git pull -q)
%cd /content/finsight
!pip install -q -r requirements.txt

In [ ]:
# Keep data, checkpoints and results on Google Drive so a disconnect doesn't lose them
# and every notebook sees the same dataset.
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/finsight/outputs /content/drive/MyDrive/finsight/results
!mkdir -p /content/drive/MyDrive/finsight/data
!rm -rf outputs results data && ln -s /content/drive/MyDrive/finsight/outputs outputs && ln -s /content/drive/MyDrive/finsight/results results && ln -s /content/drive/MyDrive/finsight/data data

In [ ]:
# The SEC asks every client to identify itself with a name and email (sent only to sec.gov).
# Store it once as a Colab secret named SEC_USER_AGENT, or type it when asked.
import os
try:
    from google.colab import userdata
    os.environ["SEC_USER_AGENT"] = userdata.get("SEC_USER_AGENT")
except Exception:
    os.environ["SEC_USER_AGENT"] = input("Your name and email for the SEC, e.g. Jane Doe jane@example.com: ")

## Quick check first: 5 companies

In [ ]:
!python -m finsight.earnings --companies 5 --out data/earnings_check

If that kept some releases, run the full build. `stats.json` explains every release that was dropped.

In [ ]:
!python -m finsight.earnings --companies 150 --out data/earnings

## Look at a few examples

In [ ]:
import json
rows = [json.loads(l) for l in open("data/earnings/train.jsonl")]
r = rows[0]
print(r["company"], r["filed"], r["url"])
print(r["target"])
print(r["text"][:3000])

## Tests (labels, scoring, API) run on CPU

In [ ]:
!pip install -q -r requirements-dev.txt && python -m pytest -q tests